# 04 · Frozen baseline and user-controlled prediction export

**Read the saved evidence, or generate your own CSV in the explicit export cell at the end.** Nothing is uploaded to Kaggle.

The tables below retain the previously frozen ranking-logistic / dynamic-Elo-logistic baseline. They are not silently replaced with a new research winner. Notebook 03 may contain a later expanded comparison; notebook 05 separates the experiments and explains its findings. The export cell uses your actual current 02/03 artifacts and the explicitly configured logistic recipe. Changing recipe families requires a supported estimator implementation and development evidence, not just editing a displayed score.

Brier is the official metric. The 2022–2025 benchmark has already been consumed; its local population includes play-ins and is not Kaggle's scored set. No 2026 tournament labels enter fitting or selection. The original competition deadline has passed; these are retrospective experiments.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.runtime import digest

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
FINAL = ROOT / "reports/final_predictions"
style()
ready = (FINAL / "evidence.json").is_file()
if ready:
    evidence = json.loads((FINAL / "evidence.json").read_text())
    for name, expected in evidence["sha256"].items():
        path = FINAL / name
        assert path.resolve().is_relative_to(FINAL.resolve())
        assert digest(path) == expected, name
    summary = json.loads((FINAL / "summary.json").read_text())
    assert summary["status"] == "completed" and not summary["future_labels_used"]
    display(Markdown(f"**{summary['rows']:,} real matchup probabilities · "
                     f"{summary['final_model_fits']} final estimators · "
                     f"training through {summary['final_training_max_season']}**"))
    display(Markdown("Submission SHA-256: `" + summary["submission_sha256"] + "`"))
else:
    display(Markdown("Current final-run evidence has not been published in this checkout. "
                     "Historical records below are not substituted for it."))

## Frozen recipe and complete matchup coverage

Men use the development-selected ranking-logistic family; women use the development-selected separate logistic family. Within each family, the feature block and regularization are selected from saved development out-of-fold predictions using mean season Brier. Identity calibration is retained; nothing is tuned on the displayed benchmark.

When both teams have tournament seeds, the seeded estimator is used. Otherwise, a separately fitted **seed-free estimator removes every seed-dependent input**. Missing seeds are not invented. All template rows retain the required lower-TeamID win-probability orientation.

In [ ]:
if ready:
    recipe = json.loads((FINAL / "recipe.json").read_text())
    fits = json.loads((FINAL / "fit_audits.json").read_text())
    table(pd.DataFrame([{"Tournament": r["gender"], "Route": r["route"],
                         "Candidate": r["candidate"]["name"],
                         "Features": len(r["features"]), "Training games": r["training_games"],
                         "Last training season": r["training_max_season"]} for r in fits]))
    table(pd.read_csv(FINAL / "routes.csv"))

## Retrospective 2022–2025 benchmark

Each season is predicted using only earlier seasons, with the same frozen recipe. The seed-free route is also evaluated on the same tournament games to expose the value of seed information. Its tournament performance does not prove generalization to teams that never qualified.

**Game-weighted Brier** averages every game's squared error. **Mean season Brier** weights seasons equally. Log loss, ROC AUC, average precision and calibration error provide complementary diagnostics. The benchmark was consumed in prior work and is not used to replace the frozen recipe.

In [ ]:
if ready:
    current = pd.read_csv(FINAL / "metrics.csv")
    seasonal = pd.read_csv(FINAL / "metrics_by_season.csv")
    table(current[["Gender", "route", "games", "brier", "mean_season_brier",
                   "log_loss", "roc_auc", "average_precision", "ece_10_bins"]])
    fig, ax = plt.subplots(figsize=(9, 4.6), constrained_layout=True)
    for (gender, route), group in seasonal.groupby(["Gender", "route"]):
        group = group.sort_values("Season")
        ax.plot(group.Season, group.brier, marker="o",
                linestyle="-" if route == "seeded" else "--",
                label=f"{'Men' if gender == 'M' else 'Women'} · {route.replace('_',' ')}")
    ax.set(title="Frozen recipe across previously consumed seasons",
           xlabel="Predicted tournament season", ylabel="Brier score · lower is better",
           xticks=[2022, 2023, 2024, 2025])
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=.2)
    ax.legend(frameon=False, ncol=2, fontsize=9)
    plt.show()

In [ ]:
if ready:
    reliability = pd.read_csv(FINAL / "reliability.csv")
    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
    ax.plot([0,1], [0,1], linestyle="--", alpha=.5, label="Perfect calibration")
    for gender, group in reliability.loc[reliability.route.eq("seeded")].groupby("Gender"):
        ax.plot(group.predicted, group.observed, marker="o",
                label=f"{'Men' if gender == 'M' else 'Women'} · seeded")
    ax.set(title="Calibration · retrospective seeded tournament forecasts",
           xlabel="Mean predicted win probability", ylabel="Observed win frequency",
           xlim=(0,1), ylim=(0,1))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=.15)
    ax.legend(frameon=False)
    plt.show()
    display(Markdown("Points summarize ten probability bins. Small bin counts and only four "
                     "seasons limit precision; proximity to the diagonal is not proof of "
                     "future calibration. No curve here is used for post-benchmark tuning."))

## Durability and release status

Each estimator and prediction batch is checkpointed only after output hashes are recorded. The private versioned S3 copy supports recovery independently of the notebook kernel or runner. The verification below starts in a fresh local run directory and requires identical CSV bytes with **zero repeated fits**.

The prediction CSV is an actual generated deliverable, but generation is not a Kaggle upload. The release records that distinction explicitly.

In [ ]:
if ready:
    recovery = json.loads((FINAL / "resume.json").read_text()) if (FINAL / "resume.json").exists() else {}
    if recovery.get("fingerprint") == summary["fingerprint"]:
        assert recovery["status"] == "passed" and recovery["repeated_fits"] == 0
        table(pd.DataFrame([{"Recovery": recovery["status"], "Tasks restored": recovery["restored_tasks"],
                             "Tasks reused": recovery["reused_tasks"], "Fits repeated": recovery["repeated_fits"],
                             "Identical CSV": recovery["identical_submission"],
                             "Kaggle upload sent": summary["kaggle_submission_sent"]}]))
    else:
        display(Markdown("Fresh-directory recovery has not yet been verified for this specific final run."))
    display(Markdown("These tables describe the stated frozen baseline, not an automatic promotion of the latest research minimum."))

## Generate and download your own file

First run **02 and 03 with `MODE = "train"`**, using your current raw inputs and matching checkpoints. Then set `GENERATE_SUBMISSION = True` below and run the cell. It calls real model refitting/inference when needed, verifies exact template order, probability bounds and team orientation, then writes **`submissions/submission.csv`** atomically. Matching successful fits are reused. It requires the existing AWS role to retain durable checkpoints.

The cell refuses stale feature or model inputs rather than exporting probabilities from an unrelated saved experiment. The recipe is explicit in `configs/inference.json`; the existing logistic recipe is a frozen baseline, not a claim that it wins every later research comparison.

Use the displayed link, or find `submissions/submission.csv` in JupyterLab's file browser and choose **Download**. You decide whether to upload it to Kaggle. There is no upload call in this workflow. This notebook does not create a CSV in review mode.

In [ ]:
from IPython.display import HTML
from march_mania.publication.workflow import generate_submission

GENERATE_SUBMISSION = False  # Change to True only when YOU are ready to generate locally.
if GENERATE_SUBMISSION:
    SUBMISSION = generate_submission(ROOT)
    display(Markdown(f"**Your file is ready:** `submissions/submission.csv`  \nSHA-256: `{digest(SUBMISSION)}`. No Kaggle upload was sent."))
    display(HTML('<a href="../submissions/submission.csv" download="submission.csv">Download your submission.csv</a>'))
else:
    display(Markdown("**Export is off.** Set `GENERATE_SUBMISSION = True` to generate and download your own file after 02/03 training."))

Sources: [official competition and evaluation](https://www.kaggle.com/competitions/march-machine-learning-mania-2026) · exact run manifests in `reports/final_predictions/` · historical [score log](../reports/submission_portfolio/). A recorded historical score is not a score from a newly rebuilt feature matrix.